In [1]:

import os
import sys
import json
import copy
import pickle
import warnings
import gc
from datetime import datetime
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from collections import defaultdict, Counter
import random
import time
from tqdm import tqdm

# Image processing
import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau, StepLR
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import timm

# Metrics and visualization
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# Kaggle datasets
import kagglehub

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set device for PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Output directory for training results
OUTPUT_DIR = Path('training_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ All libraries imported succeful")

/home/sufi/miniconda3/envs/organized/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
✅ All libraries imported succeful


In [3]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 77 — GLOBALS (run FIRST, always)                              ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, gc, torch, numpy as np, pandas as pd
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Semantic defect groups ─────────────────────────────────────────────
SEMANTIC_GROUPS = [
    'contamination', 'cut', 'deformation', 'fracture',
    'hole_void', 'minor_defect', 'scratch', 'surface_quality'
]

NUM_DEFECT_TYPES  = len(SEMANTIC_GROUPS)   # 8
NUM_PRODUCT_TYPES = 17

DEFECT_TYPE_TO_IDX = {name: idx for idx, name in enumerate(SEMANTIC_GROUPS)}
IDX_TO_DEFECT_TYPE = {idx: name for idx, name in enumerate(SEMANTIC_GROUPS)}

# ── Raw label → semantic group mapping ────────────────────────────────
DEFECT_GROUP_MAP = {
    # raw dataset labels
    'crack':               'fracture',
    'fracture':            'fracture',
    'faulty_imprint':      'fracture',
    'hole':                'hole_void',
    'void':                'hole_void',
    'pit':                 'hole_void',
    'blowhole':            'hole_void',
    'scratch':             'scratch',
    'score':               'scratch',
    'stain':               'surface_quality',
    'color':               'surface_quality',
    'rough':               'surface_quality',
    'uneven':              'surface_quality',
    'inclusion':           'surface_quality',
    'discolor':            'surface_quality',
    'pilling':             'surface_quality',
    'bent':                'deformation',
    'bent_lead':           'deformation',
    'squeeze':             'deformation',
    'deformation':         'deformation',
    'contamination':       'contamination',
    'glue':                'contamination',
    'oil':                 'contamination',
    'glue_strip':          'contamination',
    'liquid':              'contamination',
    'metal_contamination': 'contamination',
    'missing':             'minor_defect',
    'misplaced':           'minor_defect',
    'flip':                'minor_defect',
    'missing_hole':        'minor_defect',
    'thread':              'minor_defect',
    'cable_swap':          'minor_defect',
    'combined':            'minor_defect',
    'cut':                 'cut',
    # ── direct semantic group name passthroughs ────────────────────
    'hole_void':           'hole_void',
    'surface_quality':     'surface_quality',
    'minor_defect':        'minor_defect',
}

# ── Product type mapping ───────────────────────────────────────────────
PRODUCT_TYPE_TO_IDX = {}
IDX_TO_PRODUCT_TYPE = {}

print(f"✅ CELL 77 COMPLETE")
print(f"   NUM_DEFECT_TYPES  : {NUM_DEFECT_TYPES}")
print(f"   NUM_PRODUCT_TYPES : {NUM_PRODUCT_TYPES}")
print(f"   Semantic groups   : {SEMANTIC_GROUPS}")

Device: cuda
✅ CELL 77 COMPLETE
   NUM_DEFECT_TYPES  : 8
   NUM_PRODUCT_TYPES : 17
   Semantic groups   : ['contamination', 'cut', 'deformation', 'fracture', 'hole_void', 'minor_defect', 'scratch', 'surface_quality']


In [5]:
# ── AUG FIX — FIXED VERSION ───────────────────────────────────────────

CSV_PATH = '/home/sufi/merged_dataset_metadata_augmented.csv'
df_3head = pd.read_csv(CSV_PATH)

# ── Safe rename — only rename if old names exist ─────────────────────
if 'path' in df_3head.columns:
    df_3head = df_3head.rename(columns={'path': 'image_path'})
if 'category' in df_3head.columns:
    df_3head = df_3head.rename(columns={'category': 'product_type'})

print(f"Loaded CSV : {len(df_3head):,} rows")
print(f"Columns    : {list(df_3head.columns)}")

# ── Binary label ──────────────────────────────────────────────────────
def compute_binary(v):
    if pd.isna(v): return 0
    return 0 if str(v).strip().lower() in (
        '0','good','normal','ok','casting_ok') else 1

df_3head['binary_label'] = df_3head['label'].apply(compute_binary)
print(f"\n   binary_label : {df_3head['binary_label'].value_counts().to_dict()}")

# ── Defect label ──────────────────────────────────────────────────────
def remap_defect(raw):
    if pd.isna(raw): return -1
    s   = str(raw).strip().lower()
    if s in ('good','normal','casting_ok',''): return -1
    sem = DEFECT_GROUP_MAP.get(s, None)
    if sem is None and s in SEMANTIC_GROUPS: sem = s
    if sem is None:
        for k, v in DEFECT_GROUP_MAP.items():
            if k in s: sem = v; break
    return -1 if sem is None else DEFECT_TYPE_TO_IDX.get(sem, -1)

df_3head['defect_type_label'] = df_3head['defect_type'].apply(remap_defect)
df_3head.loc[df_3head['binary_label'] == 0, 'defect_type_label'] = -1

# ── Product label — use explicit loop to avoid numpy.matrix error ─────
all_products        = sorted(df_3head['product_type'].dropna().unique().tolist())
PRODUCT_TYPE_TO_IDX = {p: i for i, p in enumerate(all_products)}
IDX_TO_PRODUCT_TYPE = {i: p for i, p in enumerate(all_products)}
NUM_PRODUCT_TYPES   = len(all_products)

# Apply manually — avoids pandas .map() numpy.matrix bug
prod_labels = []
for v in df_3head['product_type']:
    if pd.isna(v):
        prod_labels.append(0)
    else:
        prod_labels.append(PRODUCT_TYPE_TO_IDX.get(str(v), 0))
df_3head['product_type_label'] = prod_labels

# ── Split ─────────────────────────────────────────────────────────────
train_df = df_3head[df_3head['split'] == 'train'].reset_index(drop=True)
val_df   = df_3head[df_3head['split'] == 'val'].reset_index(drop=True)
test_df  = df_3head[df_3head['split'] == 'test'].reset_index(drop=True)

unique = sorted(df_3head[
    df_3head['defect_type_label'] != -1
]['defect_type_label'].unique().tolist())

print(f"\n   Products       : {NUM_PRODUCT_TYPES} → {all_products}")
print(f"   Defect labels  : {unique}")
print(f"   Train          : {len(train_df):,}")
print(f"   Val            : {len(val_df):,}")
print(f"   Test           : {len(test_df):,}")

print(f"\n📊 Val samples per defect class:")
for i, name in enumerate(SEMANTIC_GROUPS):
    n = (val_df['defect_type_label'] == i).sum()
    flag = '  ⚠️ LOW' if n < 15 else ''
    print(f"   [{i}] {name:<20} val={n:>4}{flag}")

print("\n✅ AUG FIX COMPLETE")

Loaded CSV : 21,344 rows
Columns    : ['image_path', 'dataset', 'product_type', 'defect_type', 'label', 'split', 'defect_group', 'defect_type_label', 'binary_label']

   binary_label : {1: 11688, 0: 9656}

   Products       : 17 → ['bottle', 'cable', 'capsule', 'carpet', 'casting_product', 'grid', 'hazelnut', 'leather', 'magnetic_tile', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']
   Defect labels  : [0, 1, 2, 3, 4, 5, 6, 7]
   Train          : 16,337
   Val            : 3,339
   Test           : 1,668

📊 Val samples per defect class:
   [0] contamination        val=  33
   [1] cut                  val=  17
   [2] deformation          val=  15
   [3] fracture             val=  43
   [4] hole_void            val=  54
   [5] minor_defect         val=  42
   [6] scratch              val=  29
   [7] surface_quality      val=  63

✅ AUG FIX COMPLETE


In [8]:
# ============================================================
# CELL 4-LITE — EdgeNet-Lite
# Lightweight version informed by Taylor importance analysis:
#   - Smaller backbone (fewer stages = fewer redundant deep filters)
#   - Neck compressed: 320→256 (was 352→512)
#   - Defect head: 472→256→128→8 (was 520→384→192→8)
#   - Product head: 256→128→17 (was 512→256→17)
#   Target: ~40% fewer parameters than EdgeNetV4
# Copy-paste as Cell 4-Lite
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from pathlib import Path

LITE_DIR  = Path('/home/sufi/training_results/models/Lite')
LITE_DIR.mkdir(parents=True, exist_ok=True)
LITE_SAVE = LITE_DIR / 'EdgeNet_Lite_best.pth'


class CoordAttention(nn.Module):
    """
    Coordinate Attention: directional spatial context via
    H-strip and W-strip pooling. Identity-initialised.
    """
    def __init__(self, channels, reduction=16):   # reduction=16 (was 32) → richer mid
        super().__init__()
        mid = max(8, channels // reduction)
        self.conv1  = nn.Conv2d(channels, mid, 1, bias=False)
        self.bn1    = nn.BatchNorm2d(mid)
        self.act    = nn.ReLU(inplace=True)
        self.conv_h = nn.Conv2d(mid, channels, 1, bias=True)
        self.conv_w = nn.Conv2d(mid, channels, 1, bias=True)
        self._id_init()

    def _id_init(self):
        nn.init.zeros_(self.conv_h.weight);  nn.init.constant_(self.conv_h.bias, 10.)
        nn.init.zeros_(self.conv_w.weight);  nn.init.constant_(self.conv_w.bias, 10.)

    def forward(self, x):
        B, C, H, W = x.shape
        xh = x.mean(dim=3, keepdim=True)
        xw = x.mean(dim=2, keepdim=True).permute(0,1,3,2)
        y  = self.act(self.bn1(self.conv1(torch.cat([xh, xw], dim=2))))
        ah, aw = torch.split(y, [H, W], dim=2)
        aw = aw.permute(0,1,3,2)
        return x * torch.sigmoid(self.conv_h(ah)) * torch.sigmoid(self.conv_w(aw))


class EdgeNetLite(nn.Module):
    """
    EdgeNet-Lite: importance-analysis-guided lightweight variant.

    Design decisions (justified by Taylor importance analysis):
      1. Smaller backbone → importance analysis showed deep B2-only
         filters (stages 4-5) have lower gradient-weight products;
         B0 captures the high-importance early/mid features at
         half the parameter cost.
      2. Neck 256-d instead of 512-d → importance analysis showed
         the upper half of neck neurons have near-zero Taylor scores.
      3. Defect head 256→128 instead of 384→192 → second linear layer
         showed the lowest importance of all head layers.
      4. CA reduction ratio 16 (was 32) → keeps richer mid-channel
         attention since CA layers showed HIGH importance scores.
    """
    def __init__(self, num_defect_classes=8, num_product_classes=17):
        super().__init__()

        # ── Backbone: B0 — pretrained MBConv feature extractor ───────
        # Taylor analysis showed stages 2-4 dominate importance;
        # B0 covers these at 3.6M params vs B2's 7.7M.
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=True,
            features_only=True,
            out_indices=(2, 3, 4),          # 40ch@s8 | 112ch@s16 | 320ch@s32
        )
        chs = self.backbone.feature_info.channels()   # [40, 112, 320]
        c2, c3, c4 = chs[0], chs[1], chs[2]

        # ── CoordAttention (kept — showed HIGH importance scores) ─────
        self.ca_mid  = CoordAttention(c3, reduction=16)
        self.ca_deep = CoordAttention(c4, reduction=16)

        # ── Compressed shared neck: 320→256 ──────────────────────────
        NECK_DIM = 256
        self.neck = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(c4, NECK_DIM, bias=False),
            nn.BatchNorm1d(NECK_DIM),
            nn.SiLU(),
            nn.Dropout(0.30),
        )

        # ── Binary head ────────────────────────────────────────────────
        self.binary_head = nn.Linear(NECK_DIM, 1)

        # ── Product head: 256→128→17 ───────────────────────────────────
        self.product_head = nn.Sequential(
            nn.Linear(NECK_DIM, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_product_classes),
        )

        # ── Compressed multi-scale defect head: 472→256→128→8 ─────────
        # Taylor: second FC (192→8 in V4) had lowest head importance.
        # Fix: go 256→128→8 with more dropout to compensate capacity.
        self.defect_gap  = nn.AdaptiveAvgPool2d(1)
        defect_in        = c2 + c3 + c4                      # 40+112+320 = 472
        self.defect_head = nn.Sequential(
            nn.Linear(defect_in, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.35),
            nn.Linear(256, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_defect_classes),
        )

        self._init_heads()
        self.ca_mid._id_init()
        self.ca_deep._id_init()

    def _init_heads(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
                nn.init.ones_(m.weight);  nn.init.zeros_(m.bias)

    def forward(self, x):
        f2, f3, f4 = self.backbone(x)      # 40@s8 | 112@s16 | 320@s32
        f3 = self.ca_mid(f3)
        f4 = self.ca_deep(f4)
        shared  = self.neck(f4)            # [B, 256]
        bin_out = self.binary_head(shared) # [B, 1]
        prod_out= self.product_head(shared)# [B, 17]
        d2 = self.defect_gap(f2).flatten(1)
        d3 = self.defect_gap(f3).flatten(1)
        d4 = self.defect_gap(f4).flatten(1)
        def_out = self.defect_head(torch.cat([d2, d3, d4], dim=1))
        return bin_out, def_out, prod_out

    def count_params(self):
        total = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total / 1e6


# ── Build + sanity check ───────────────────────────────────────
lite_model = EdgeNetLite(
    num_defect_classes  = NUM_DEFECT_TYPES,
    num_product_classes = NUM_PRODUCT_TYPES,
).to(device)

lite_model.eval()
with torch.no_grad():
    _d = torch.randn(2, 3, 224, 224).to(device)
    _b, _def, _p = lite_model(_d)
    assert _b.shape   == (2, 1)
    assert _def.shape == (2, NUM_DEFECT_TYPES)
    assert _p.shape   == (2, NUM_PRODUCT_TYPES)

    _chs = lite_model.backbone.feature_info.channels()
    for ca, ch, tag in [
        (lite_model.ca_mid,  _chs[1], f'ca_mid  ({_chs[1]}ch)'),
        (lite_model.ca_deep, _chs[2], f'ca_deep ({_chs[2]}ch)'),
    ]:
        _t  = torch.ones(1, ch, 14, 14).to(device)
        _r  = (ca(_t) / _t).mean().item()
        _ok = '✅' if abs(_r-1.0) < 0.01 else '❌'
        print(f"   {tag} identity: {_r:.6f}  {_ok}")
    del _d, _b, _def, _p, _t

print(f"\n✅ CELL 4-LITE COMPLETE — EdgeNet-Lite")
print(f"   Backbone   : EfficientNet-B0  (40+112+320 ch)")
print(f"   Neck       : 320 → 256")
print(f"   Defect head: 472 → 256 → 128 → {NUM_DEFECT_TYPES}")
print(f"   Product    : 256 → 128 → {NUM_PRODUCT_TYPES}")
print(f"   Total params (trainable): {lite_model.count_params():.2f}M")
print(f"   Save path  : {LITE_SAVE}")

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


   ca_mid  (112ch) identity: 0.999909  ✅
   ca_deep (320ch) identity: 0.999909  ✅

✅ CELL 4-LITE COMPLETE — EdgeNet-Lite
   Backbone   : EfficientNet-B0  (40+112+320 ch)
   Neck       : 320 → 256
   Defect head: 472 → 256 → 128 → 8
   Product    : 256 → 128 → 17
   Total params (trainable): 3.89M
   Save path  : /home/sufi/training_results/models/Lite/EdgeNet_Lite_best.pth


In [9]:
# ============================================================
# CELL 4.5 CORRECTED — Transfer backbone + neck + binary only
#
# Why defect head is NOT transferred:
#   V4 defect_head.8 (192→8) was trained to receive inputs from
#   V4's defect_head.4 (384→192 projection).
#   Lite's defect_head.4 receives from a different 520→192 space.
#   Transplanting that final linear into the wrong input space
#   produces noise — proven by defect=0.0% vs binary=100%.
#
# What we DO transfer (7.2M params):
#   backbone.*  → exact copy (identical B2 arch)
#   neck.2/3.*  → first 256 rows/elements of V4's 512-wide layers
#   binary_head → first 256 cols (binary=100% sanity check proved this)
#
# Defect + product heads: xavier re-init (small, trains fast).
# With perfect backbone features, expect F1>0.55 by epoch 3.
# ============================================================

from pathlib import Path
import torch, torch.nn as nn

V4_CKPT = Path('/home/sufi/training_results/models/V4/EdgeNet_V4_best.pth')
v4_ckpt = torch.load(V4_CKPT, map_location='cpu')
v4_sd   = v4_ckpt['model_state_dict']
print(f"Loaded V4 checkpoint — epoch {v4_ckpt['epoch']}, F1={v4_ckpt['score']:.4f}")

lite_sd = lite_model.state_dict()

transferred_params = 0
skipped_params     = 0
backbone_count     = 0

for key in lite_sd.keys():
    lt = lite_sd[key]

    # ── Backbone: exact same B2 arch ─────────────────────────
    if key.startswith('backbone.'):
        if key in v4_sd and v4_sd[key].shape == lt.shape:
            lite_sd[key] = v4_sd[key].clone()
            transferred_params += lt.numel()
            backbone_count += 1
        else:
            skipped_params += lt.numel()

    # ── Neck linear (352→256): first 256 rows of [512,352] ───
    elif key == 'neck.2.weight':
        lite_sd[key] = v4_sd['neck.2.weight'][:256, :].clone()
        transferred_params += lt.numel()

    # ── Neck BN: first 256 of 512 ────────────────────────────
    elif key in ('neck.3.weight', 'neck.3.bias',
                 'neck.3.running_mean', 'neck.3.running_var'):
        lite_sd[key] = v4_sd[key][:256].clone()
        transferred_params += lt.numel()
    elif key == 'neck.3.num_batches_tracked':
        lite_sd[key] = v4_sd.get(key, lt).clone()

    # ── Binary head: first 256 cols of [1,512] ───────────────
    elif key == 'binary_head.weight':
        lite_sd[key] = v4_sd['binary_head.weight'][:, :256].clone()
        transferred_params += lt.numel()
    elif key == 'binary_head.bias':
        lite_sd[key] = v4_sd['binary_head.bias'].clone()
        transferred_params += lt.numel()

    # ── Defect head: DO NOT transfer (wrong intermediate space)
    # ── Product head: DO NOT transfer (different dims)
    else:
        skipped_params += lt.numel()

lite_model.load_state_dict(lite_sd, strict=True)

# ── Re-initialise defect + product heads cleanly ─────────────
def _xavier_init(module):
    for m in module.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.BatchNorm1d):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

_xavier_init(lite_model.defect_head)
_xavier_init(lite_model.product_head)
print("  Defect + product heads re-initialised (xavier_uniform)")

print(f"\n{'='*60}")
print(f"  WEIGHT TRANSFER COMPLETE")
print(f"  Backbone layers : {backbone_count} tensors  ({transferred_params/1e6:.2f}M params)")
print(f"  Total transferred: {transferred_params/1e6:.3f}M")
print(f"  Re-initialised  : {skipped_params/1e6:.3f}M  (defect/product heads)")
print(f"  Coverage (backbone+neck+binary): {100*transferred_params/(transferred_params+skipped_params):.1f}%")
print(f"{'='*60}")

# ── Sanity check ─────────────────────────────────────────────
print("\n  Sanity check (1 val batch)...")
lite_model.eval()
with torch.no_grad():
    for imgs, bl, dl, pl, _ in val_3head_loader:
        imgs, bl, dl, pl = imgs.to(device), bl.to(device), dl.to(device), pl.to(device)
        sb, sd, sp = lite_model(imgs); sb_sq = sb.squeeze(1)
        bin_acc  = ((torch.sigmoid(sb_sq)>0.5).long()==bl).float().mean().item()
        mk = dl != -1
        def_acc  = (sd.argmax(1)[mk]==dl[mk]).float().mean().item() if mk.sum()>0 else 0
        prod_acc = (sp.argmax(1)==pl).float().mean().item()
        print(f"  Binary={bin_acc*100:.1f}%  DefectType={def_acc*100:.1f}%  Product={prod_acc*100:.1f}%")
        print()
        print(f"  ✅ Binary ≈ 100%  → backbone + binary head transfer PERFECT")
        print(f"  ✅ DefectType ≈ {100/NUM_DEFECT_TYPES:.0f}%  → random init (expected before training)")
        print(f"  ✅ Product ≈ {100/NUM_PRODUCT_TYPES:.0f}%  → random init (expected before training)")
        print(f"\n  Backbone is perfect. Heads train fast from here.")
        print(f"  Expected by epoch 3: Defect F1 > 0.55 (same as V4 epoch 3)")
        break

Loaded V4 checkpoint — epoch 100, F1=0.9541


RuntimeError: Error(s) in loading state_dict for EdgeNetLite:
	size mismatch for neck.2.weight: copying a param with shape torch.Size([256, 352]) from checkpoint, the shape in current model is torch.Size([256, 320]).

In [6]:
# ════════════════════════════════════════════════════════════════════════
# EDGENETV4 — GATED INFERENCE + EVALUATION (Complete Drop-in)
# ════════════════════════════════════════════════════════════════════════
# Contains:
#   1. predict_with_gating()       — gated forward pass (defect = -1 if GOOD)
#   2. format_prediction()         — human-readable output
#   3. compute_defect_metrics_gated() — gated per-class metrics
#   4. validate_epoch_gated()      — gated validation loop
#   5. evaluate_test_set_gated()   — gated test-set evaluation
#   6. Prediction Checker v3       — gated visualisation
#
# Requires from previous cells:
#   model, device, NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES, SEMANTIC_GROUPS
#   val_3head_loader, test_3head_loader (optional)
#   EMA_START (from training script)
# ════════════════════════════════════════════════════════════════════════

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support


# ════════════════════════════════════════════════════════════════════════
# 1. GATED INFERENCE
# ════════════════════════════════════════════════════════════════════════

def predict_with_gating(model, images, device, binary_threshold=0.5):
    """
    Run all 3 heads, then gate defect prediction by binary outcome.

    If binary head predicts GOOD (0), defect prediction is forced to -1
    (N/A). This matches training where normal samples have
    defect_type_label=-1 and are excluded from the defect loss.

    Args:
        model            : EdgeNetV4 (eval mode)
        images           : [B, 3, H, W] tensor (already normalised)
        device           : cuda / cpu
        binary_threshold : sigmoid threshold (default 0.5)

    Returns:
        dict with:
            binary_pred  [B] long   — 0=GOOD, 1=DEFECT
            binary_prob  [B] float  — sigmoid probability
            defect_pred  [B] long   — class idx 0..7, or -1 if GOOD
            defect_conf  [B] float  — softmax max confidence (0 if GOOD)
            product_pred [B] long   — class idx 0..16
            product_conf [B] float  — softmax max confidence
    """
    model.eval()
    with torch.no_grad():
        bin_out, def_out, prod_out = model(images.to(device))

        # ── Binary head ───────────────────────────────────────
        bin_prob = torch.sigmoid(bin_out.squeeze(1))
        bin_pred = (bin_prob > binary_threshold).long()

        # ── Defect head (raw) ─────────────────────────────────
        def_softmax  = torch.softmax(def_out, dim=1)
        def_pred_raw = def_softmax.argmax(1)
        def_conf_raw = def_softmax.max(1).values

        # ── GATE: suppress defect when binary says GOOD ───────
        sentinel = torch.full_like(def_pred_raw, -1)
        def_pred = torch.where(bin_pred == 1, def_pred_raw, sentinel)
        def_conf = torch.where(
            bin_pred == 1, def_conf_raw, torch.zeros_like(def_conf_raw)
        )

        # ── Product head (never gated) ────────────────────────
        prod_softmax = torch.softmax(prod_out, dim=1)
        prod_pred    = prod_softmax.argmax(1)
        prod_conf    = prod_softmax.max(1).values

    return {
        'binary_pred':  bin_pred.cpu(),
        'binary_prob':  bin_prob.cpu(),
        'defect_pred':  def_pred.cpu(),
        'defect_conf':  def_conf.cpu(),
        'product_pred': prod_pred.cpu(),
        'product_conf': prod_conf.cpu(),
    }


def format_prediction(bin_pred, def_pred, prod_pred,
                      idx_to_defect, idx_to_product):
    """Human-readable prediction strings for a single sample."""
    bin_str  = 'DEFECT' if bin_pred == 1 else 'GOOD'
    prod_str = idx_to_product.get(
        str(prod_pred), idx_to_product.get(prod_pred, f'prod_{prod_pred}')
    )
    if def_pred == -1:
        def_str = 'N/A (no defect)'
    else:
        def_str = idx_to_defect.get(
            str(def_pred), idx_to_defect.get(def_pred, f'def_{def_pred}')
        )
    return {'binary': bin_str, 'defect': def_str, 'product': prod_str}


# ════════════════════════════════════════════════════════════════════════
# 2. GATED EVALUATION METRICS
# ════════════════════════════════════════════════════════════════════════

def compute_defect_metrics_gated(bin_preds, bin_labels,
                                 def_preds, def_labels,
                                 num_classes=None):
    """
    Per-class defect metrics, evaluated ONLY on samples where:
      1. True label is DEFECT (def_labels != -1)
      2. AND binary head predicted DEFECT (bin_preds == 1)

    This matches the gated inference pipeline: defect type is only
    displayed when binary says DEFECT, so metrics should reflect that.

    Args:
        bin_preds   [N] np.array — predicted binary (0/1)
        bin_labels  [N] np.array — true binary     (0/1)
        def_preds   [N] np.array — predicted defect (0..7 or -1)
        def_labels  [N] np.array — true defect      (0..7 or -1)
        num_classes  int          — number of defect classes

    Returns:
        dict: f1, prec, rec, per_p, per_r, per_f, n_evaluated
    """
    nc = num_classes or NUM_DEFECT_TYPES

    # ── Gated mask: only true defects where binary said DEFECT ──
    valid_mask = (def_labels != -1) & (bin_preds == 1)
    n_evaluated = int(valid_mask.sum())

    if n_evaluated == 0:
        z = np.zeros(nc)
        return {'f1': 0.0, 'prec': 0.0, 'rec': 0.0,
                'per_p': z, 'per_r': z, 'per_f': z,
                'n_evaluated': 0}

    p, r, f, _ = precision_recall_fscore_support(
        def_labels[valid_mask], def_preds[valid_mask],
        average=None, labels=list(range(nc)), zero_division=0
    )
    return {
        'f1':          float(f.mean()),
        'prec':        float(p.mean()),
        'rec':         float(r.mean()),
        'per_p':       p,
        'per_r':       r,
        'per_f':       f,
        'n_evaluated': n_evaluated,
    }


# ════════════════════════════════════════════════════════════════════════
# 3. GATED VALIDATION LOOP
# ════════════════════════════════════════════════════════════════════════

def validate_epoch_gated(model, loader, ema, epoch, device,
                         ema_start=5):
    """
    Validation with gated inference.
    Binary, product accuracy are computed normally (always meaningful).
    Defect metrics are gated by binary outcome.
    """
    use_ema = (ema is not None) and (epoch >= ema_start)
    if use_ema:
        ema.apply(model)

    model.eval()
    tot = bin_c = prod_c = 0
    all_bin_pred, all_bin_true   = [], []
    all_def_pred, all_def_true   = [], []
    all_prod_pred, all_prod_true = [], []

    with torch.no_grad():
        for imgs, bin_lbl, def_lbl, prod_lbl, _ in loader:
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)

            # Gated forward pass — defect pred = -1 if binary says GOOD
            out      = predict_with_gating(model, imgs, device)
            bin_pred = out['binary_pred'].to(device)
            def_pred = out['defect_pred'].to(device)
            prod_pred= out['product_pred'].to(device)

            tot    += imgs.size(0)
            bin_c  += (bin_pred  == bin_lbl ).sum().item()
            prod_c += (prod_pred == prod_lbl).sum().item()

            all_bin_pred.append(bin_pred.cpu().numpy())
            all_bin_true.append(bin_lbl.cpu().numpy())
            all_def_pred.append(def_pred.cpu().numpy())
            all_def_true.append(def_lbl.cpu().numpy())
            all_prod_pred.append(prod_pred.cpu().numpy())
            all_prod_true.append(prod_lbl.cpu().numpy())

    if use_ema:
        ema.restore(model)

    # Concatenate
    bin_pred_all  = np.concatenate(all_bin_pred)
    bin_true_all  = np.concatenate(all_bin_true)
    def_pred_all  = np.concatenate(all_def_pred)
    def_true_all  = np.concatenate(all_def_true)
    prod_pred_all = np.concatenate(all_prod_pred)
    prod_true_all = np.concatenate(all_prod_true)

    # ── Gated defect metrics ──
    m = compute_defect_metrics_gated(
        bin_pred_all, bin_true_all,
        def_pred_all, def_true_all
    )

    # ── Defect accuracy on truly-defective samples (gated) ──
    valid_def = (def_true_all != -1) & (bin_pred_all == 1)
    if valid_def.sum() > 0:
        def_acc = 100. * (def_pred_all[valid_def] ==
                          def_true_all[valid_def]).sum() / valid_def.sum()
    else:
        def_acc = 0.0

    return {
        'binary_acc':  100. * bin_c  / tot,
        'defect_acc':  def_acc,
        'product_acc': 100. * prod_c / tot,
        'val_f1':      m['f1'],
        'val_per_p':   m['per_p'],
        'val_per_r':   m['per_r'],
        'val_per_f':   m['per_f'],
        'n_evaluated': m['n_evaluated'],
    }


# ════════════════════════════════════════════════════════════════════════
# 4. GATED TEST-SET EVALUATION (with full classification report)
# ════════════════════════════════════════════════════════════════════════

def evaluate_test_set_gated(model, loader, device, class_names=None):
    """
    Full test-set evaluation with gated inference.
    Prints binary, product, and per-class defect metrics.

    Args:
        model       : EdgeNetV4 in eval mode (weights already loaded)
        loader      : test DataLoader (returns img, bin, def, prod, path)
        device      : cuda / cpu
        class_names : list of defect class names (defaults to SEMANTIC_GROUPS)

    Returns:
        dict of full metrics
    """
    names = class_names or SEMANTIC_GROUPS

    model.eval()
    tot = bin_c = prod_c = 0
    all_bin_pred, all_bin_true   = [], []
    all_def_pred, all_def_true   = [], []
    all_prod_pred, all_prod_true = [], []

    print(f"\n🔍 Evaluating test set ({len(loader.dataset)} samples) — "
          f"GATED inference\n")

    with torch.no_grad():
        for imgs, bin_lbl, def_lbl, prod_lbl, _ in loader:
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)

            out      = predict_with_gating(model, imgs, device)
            bin_pred = out['binary_pred'].to(device)
            def_pred = out['defect_pred'].to(device)
            prod_pred= out['product_pred'].to(device)

            tot    += imgs.size(0)
            bin_c  += (bin_pred  == bin_lbl ).sum().item()
            prod_c += (prod_pred == prod_lbl).sum().item()

            all_bin_pred.append(bin_pred.cpu().numpy())
            all_bin_true.append(bin_lbl.cpu().numpy())
            all_def_pred.append(def_pred.cpu().numpy())
            all_def_true.append(def_lbl.cpu().numpy())
            all_prod_pred.append(prod_pred.cpu().numpy())
            all_prod_true.append(prod_lbl.cpu().numpy())

    bin_pred_all  = np.concatenate(all_bin_pred)
    bin_true_all  = np.concatenate(all_bin_true)
    def_pred_all  = np.concatenate(all_def_pred)
    def_true_all  = np.concatenate(all_def_true)
    prod_pred_all = np.concatenate(all_prod_pred)
    prod_true_all = np.concatenate(all_prod_true)

    m = compute_defect_metrics_gated(
        bin_pred_all, bin_true_all,
        def_pred_all, def_true_all,
        num_classes=len(names)
    )

    valid_def = (def_true_all != -1) & (bin_pred_all == 1)
    if valid_def.sum() > 0:
        def_acc = 100. * (def_pred_all[valid_def] ==
                          def_true_all[valid_def]).sum() / valid_def.sum()
    else:
        def_acc = 0.0

    bin_acc  = 100. * bin_c  / tot
    prod_acc = 100. * prod_c / tot

    # ── Print report ──────────────────────────────────────────
    print(f"{'='*60}")
    print(f"  GATED TEST-SET RESULTS")
    print(f"{'='*60}")
    print(f"  Total samples         : {tot:,}")
    print(f"  Binary accuracy       : {bin_acc:.2f}%")
    print(f"  Product accuracy      : {prod_acc:.2f}%")
    print(f"  Defect accuracy*      : {def_acc:.2f}%")
    print(f"  Defect F1 (macro)*    : {m['f1']:.4f}")
    print(f"  Samples evaluated*    : {m['n_evaluated']} / "
          f"{(def_true_all != -1).sum()} true defects")
    print(f"  * Defect metrics gated by binary outcome")

    # ── How many true defects did binary miss? ────────────────
    true_def_mask     = (def_true_all != -1)
    bin_missed        = (true_def_mask & (bin_pred_all == 0)).sum()
    bin_caught        = (true_def_mask & (bin_pred_all == 1)).sum()
    print(f"\n  Binary head on defects:")
    print(f"    Caught (correctly flagged) : {bin_caught}")
    print(f"    Missed (said GOOD)         : {bin_missed}")

    # ── Per-class table ───────────────────────────────────────
    print(f"\n  {'Class':<22} {'P':>7} {'R':>7} {'F1':>7}")
    print(f"  {'─'*46}")
    for i, name in enumerate(names):
        print(f"  {name:<22} "
              f"{m['per_p'][i]:>7.3f} "
              f"{m['per_r'][i]:>7.3f} "
              f"{m['per_f'][i]:>7.3f}")
    print(f"  {'─'*46}")
    print(f"  {'MACRO':<22} "
          f"{m['per_p'].mean():>7.3f} "
          f"{m['per_r'].mean():>7.3f} "
          f"{m['per_f'].mean():>7.3f}")
    print(f"{'='*60}\n")

    return {
        'binary_acc':  bin_acc,
        'defect_acc':  def_acc,
        'product_acc': prod_acc,
        'f1_macro':    m['f1'],
        'per_p':       m['per_p'],
        'per_r':       m['per_r'],
        'per_f':       m['per_f'],
        'n_evaluated': m['n_evaluated'],
        'n_total_defects': int((def_true_all != -1).sum()),
        'bin_missed_defects': int(bin_missed),
    }


# ════════════════════════════════════════════════════════════════════════
# 5. PREDICTION CHECKER v3 — Gated Visualisation
# ════════════════════════════════════════════════════════════════════════

def run_prediction_checker_gated(
    model, loader, device,
    ckpt_path=None,
    save_path=None,
    n_wrong=6, n_total=12, seed=99,
):
    """
    Visual prediction checker with gated inference.

    Sampling logic guarantees you see failures (n_wrong wrong predictions
    interleaved with correct ones) so the grid isn't all green.

    Args:
        model     : EdgeNetV4 (weights already loaded)
        loader    : DataLoader (typically val_3head_loader)
        device    : cuda / cpu
        ckpt_path : optional path to checkpoint for label maps
        save_path : where to save the figure (None = don't save)
        n_wrong   : how many wrong samples to force into the grid
        n_total   : total grid size (must be 12 for 3x4 layout)
        seed      : RNG seed for sampling
    """
    # ── Label maps ────────────────────────────────────────────
    if ckpt_path is not None and Path(ckpt_path).exists():
        ckpt    = torch.load(ckpt_path, map_location=device,
                             weights_only=False)
        idx2def = ckpt.get('idx_to_defect_type',
                           {str(i): n for i, n in enumerate(SEMANTIC_GROUPS)})
        idx2prod= ckpt.get('idx_to_product_type',
                           {str(i): f'product_{i}'
                            for i in range(NUM_PRODUCT_TYPES)})
    else:
        idx2def  = {str(i): n for i, n in enumerate(SEMANTIC_GROUPS)}
        idx2prod = {str(i): f'product_{i}'
                    for i in range(NUM_PRODUCT_TYPES)}

    def def_name(idx):
        if idx == -1: return 'N/A (no defect)'
        return idx2def.get(str(idx), idx2def.get(idx, f'def_{idx}'))

    def prod_name(idx):
        return idx2prod.get(str(idx),
                            idx2prod.get(idx, f'prod_{idx}'))

    # ── Denormalise ───────────────────────────────────────────
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def denorm(t):
        img = (t.cpu() * STD + MEAN).clamp(0, 1)
        return (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

    # ── Collect samples ───────────────────────────────────────
    pool_correct, pool_wrong = [], []

    print("Scanning loader with gated inference...")
    for imgs, bin_lbl, def_lbl, prod_lbl, _ in loader:
        out = predict_with_gating(model, imgs, device)

        for i in range(imgs.size(0)):
            b_true = int(bin_lbl[i].item())
            b_pred = int(out['binary_pred'][i].item())
            d_true = int(def_lbl[i].item())
            d_pred = int(out['defect_pred'][i].item())
            p_true = int(prod_lbl[i].item())
            p_pred = int(out['product_pred'][i].item())

            rec = (imgs[i], b_true, b_pred,
                   d_true, d_pred, p_true, p_pred)

            # ── Correctness checks ──
            bin_wrong = (b_true != b_pred)
            # Defect correctness with gating:
            if d_true == -1 and d_pred == -1:
                def_wrong = False                  # both agree: no defect
            elif d_true == -1 and d_pred != -1:
                def_wrong = True                   # good sample but predicted defect type
            elif d_true != -1 and d_pred == -1:
                def_wrong = True                   # missed: binary said good but truly defect
            else:
                def_wrong = (d_true != d_pred)     # both defective: class match?
            prod_wrong = (p_true != p_pred)

            if bin_wrong or def_wrong or prod_wrong:
                pool_wrong.append(rec)
            else:
                pool_correct.append(rec)

        if len(pool_wrong) >= 40 and len(pool_correct) >= 40:
            break

    print(f"  {len(pool_wrong)} wrong | {len(pool_correct)} correct")

    # ── Sample 12 (interleave wrong + correct) ────────────────
    rng = np.random.default_rng(seed=seed)
    nw  = min(n_wrong, len(pool_wrong))
    nc  = n_total - nw

    wrong_sel   = [pool_wrong[i]   for i in rng.choice(
        len(pool_wrong),   size=nw, replace=False)] if nw > 0 else []
    correct_sel = [pool_correct[i] for i in rng.choice(
        len(pool_correct), size=nc, replace=False)] if nc > 0 else []

    samples = []
    wi = ci = 0
    for slot in range(n_total):
        if slot % 2 == 0 and wi < len(wrong_sel):
            samples.append(wrong_sel[wi]); wi += 1
        elif ci < len(correct_sel):
            samples.append(correct_sel[ci]); ci += 1
        elif wi < len(wrong_sel):
            samples.append(wrong_sel[wi]); wi += 1

    # ── Plot ──────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 4, figsize=(22, 16))
    fig.patch.set_facecolor('#1C1C1E')
    fig.suptitle(
        'EdgeNetV4 — Prediction Checker (Binary-Gated)\n'
        'Defect type suppressed to N/A when binary predicts GOOD',
        fontsize=13, fontweight='bold', color='white', y=1.005)

    for ax, rec in zip(axes.flat, samples):
        img_t, b_true, b_pred, d_true, d_pred, p_true, p_pred = rec

        b_ok = (b_true == b_pred)
        # Defect correctness (matches the logic above)
        if d_true == -1 and d_pred == -1:
            d_ok = True
        elif d_true == -1 or d_pred == -1:
            d_ok = False
        else:
            d_ok = (d_true == d_pred)
        p_ok = (p_true == p_pred)
        all_ok = b_ok and d_ok and p_ok

        border = '#2DC653' if all_ok else '#E63946'

        ax.imshow(denorm(img_t))
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(border)
            spine.set_linewidth(6)

        # ── Binary ──
        b_true_str = 'DEFECT' if b_true == 1 else 'GOOD'
        b_pred_str = 'DEFECT' if b_pred == 1 else 'GOOD'
        b_mark = '✓' if b_ok else '✗'
        b_col  = '#7FFF7F' if b_ok else '#FF6B6B'

        # ── Defect ──
        d_true_str = def_name(d_true)
        d_pred_str = def_name(d_pred)
        d_mark = '✓' if d_ok else '✗'
        d_col  = '#7FFF7F' if d_ok else '#FF6B6B'

        # ── Product ──
        p_true_str = prod_name(p_true)
        p_pred_str = prod_name(p_pred)
        p_mark = '✓' if p_ok else '✗'
        p_col  = '#7FFF7F' if p_ok else '#FF6B6B'

        lines = [
            ('Binary',      b_mark, b_true_str, b_pred_str, b_col),
            ('Defect Type', d_mark, d_true_str, d_pred_str, d_col),
            ('Product',     p_mark, p_true_str, p_pred_str, p_col),
        ]

        y_start = -0.04
        ax.text(0.0, y_start,
                f"{'Head':<12}  {'✓✗'}  {'True':<20}  Pred",
                transform=ax.transAxes, fontsize=6.5,
                color='#CCCCCC', fontfamily='monospace')
        ax.text(0.0, y_start - 0.06, '─'*55,
                transform=ax.transAxes, fontsize=6, color='#555555',
                fontfamily='monospace')

        for j, (head, mark, true_s, pred_s, col) in enumerate(lines):
            y = y_start - 0.13 - j * 0.09
            ax.text(0.0, y,
                    f"{head:<12}  {mark}   {true_s:<20}  {pred_s}",
                    transform=ax.transAxes, fontsize=6.8,
                    color=col, fontfamily='monospace', fontweight='bold')

    c_patch = mpatches.Patch(color='#2DC653', label='All 3 heads correct')
    w_patch = mpatches.Patch(color='#E63946', label='At least 1 head wrong')
    fig.legend(handles=[c_patch, w_patch], loc='lower center', ncol=2,
               fontsize=11, labelcolor='white',
               facecolor='#2C2C2E', edgecolor='gray',
               bbox_to_anchor=(0.5, -0.04))

    plt.subplots_adjust(hspace=0.55, wspace=0.1)

    if save_path is not None:
        plt.savefig(save_path, dpi=160, bbox_inches='tight',
                    facecolor='#1C1C1E')
        print(f"  Saved: {save_path}")
    plt.show()

    # ── Console summary ───────────────────────────────────────
    print("\n  ── Prediction Summary (Gated) ─────────────────────")
    print(f"  {'#':<3}  {'Bin':<5}  {'Def':<5}  {'Prod':<5}  "
          f"{'True Def':<22} → Pred Def")
    print(f"  {'─'*70}")
    for k, (img_t, bt, bp, dt, dp, pt, pp) in enumerate(samples):
        bm = '✓' if bt == bp else '✗'
        if dt == -1 and dp == -1:
            dm = '✓'
        elif dt == -1 or dp == -1:
            dm = '✗'
        else:
            dm = '✓' if dt == dp else '✗'
        pm = '✓' if pt == pp else '✗'
        print(f"  {k+1:<3}  Bin:{bm}  Def:{dm}  Prod:{pm}  "
              f"{def_name(dt):<22} → {def_name(dp)}")

    return samples


# ════════════════════════════════════════════════════════════════════════
# 6. RUN IT — Load checkpoint and evaluate
# ════════════════════════════════════════════════════════════════════════

CKPT_PATH = Path('/home/sufi/training_results/models/V4/EdgeNet_V4_best.pth')
SAVE_DIR  = Path('/home/sufi/training_results/models/V4')

# ── Load checkpoint ──────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"✅ Loaded checkpoint — epoch {ckpt['epoch']}, "
      f"Val F1 = {ckpt['score']:.4f}\n")

# ── Run gated test-set evaluation (if test loader exists) ────
if 'test_3head_loader' in globals():
    test_metrics = evaluate_test_set_gated(
        model, test_3head_loader, device,
        class_names=SEMANTIC_GROUPS
    )
else:
    print("⚠️  test_3head_loader not found — running on val_3head_loader instead\n")
    test_metrics = evaluate_test_set_gated(
        model, val_3head_loader, device,
        class_names=SEMANTIC_GROUPS
    )

# ── Visual prediction checker ────────────────────────────────
samples = run_prediction_checker_gated(
    model,
    loader    = val_3head_loader,
    device    = device,
    ckpt_path = CKPT_PATH,
    save_path = SAVE_DIR / 'prediction_checker_v3_gated.png',
    n_wrong   = 6,
    n_total   = 12,
    seed      = 99,
)

print("\n✅ ALL GATED EVALUATION COMPLETE")

NameError: name 'model' is not defined